# VAE Datasets

> datasets designed to work specifically with VAE models.

In [ ]:
#| default_exp data.vae_datasets

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import torch
from torch import Tensor
from pathlib import Path
from typing import List, Optional, Sequence, Union, Any, Callable
from torchvision.datasets.folder import default_loader
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import CelebA
import torchvision.transforms.v2 as transforms
import torch.nn.functional as F
import zipfile
import h5py
import numpy as np


class MultiAgentPOVDataset(Dataset):

    def __init__(self, h5_path, split="train", val_ratio=0.1, transform=None):
        """Args:

        h5_path (str): Path to the h5 dataset file.
        split (str): 'train' or 'val' to specify the dataset split.
        val_ratio (float): Fraction of episodes to allocate to the validation
        set.
        transform (callable, optional): Unified transform pipeline.
        """
        assert split in ["train", "val"], "split must be 'train' or 'val'"
        self.h5_path = h5_path
        self.split = split
        self.transform = transform

        # Initialized as None for PyTorch multi-process DataLoader safety
        self.file = None
        self.index_map = []

        # Open temporarily to inspect structure and build split mappings
        with h5py.File(self.h5_path, "r") as f:
            # 1. Gather all episode keys and sort them so the split is deterministic
            all_episodes = sorted(list(f.keys()))
            num_episodes = len(all_episodes)

            # 2. Calculate the split boundary index
            val_count = int(num_episodes * val_ratio)
            train_count = num_episodes - val_count

            # 3. Partition the actual episode groups (Episode-level splitting)
            if self.split == "train":
                active_episodes = all_episodes[:train_count]
            else:
                active_episodes = all_episodes[train_count:]

            # 4. Build the index map *only* using the active episodes for this split
            for episode_key in active_episodes:
                episode_group = f[episode_key]
                num_frames = np.min(episode_group['agents_success_at']).item()#episode_group["0_pov"].shape[0]

                for frame_idx in range(num_frames):
                    self.index_map.append((episode_key, frame_idx, "0_pov"))
                    self.index_map.append((episode_key, frame_idx, "1_pov"))

        print(
            f"Initialized {self.split} split with {len(active_episodes)} episodes ({len(self.index_map)} total agent frames)."
        )

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        if self.file is None:
            self.file = h5py.File(self.h5_path, "r")

        episode_key, frame_idx, agent_pov_key = self.index_map[idx]
        frame = self.file[episode_key][agent_pov_key][frame_idx]

        if self.transform:
            frame = self.transform(frame)

        return frame

    def close(self):
        if self.file is not None:
            self.file.close()
            self.file = None

class VAEDataset(LightningDataModule):
    """
    PyTorch Lightning data module 

    Args:
        data_dir: root directory of your dataset.
        train_batch_size: the batch size to use during training.
        val_batch_size: the batch size to use during validation.
        patch_size: the size of the crop to take from the original images.
        num_workers: the number of parallel workers to create to load data
            items (see PyTorch's Dataloader documentation for more details).
        pin_memory: whether prepared items should be loaded into pinned memory
            or not. This can improve performance on GPUs.
    """

    def __init__(
        self,
        data_path: str,
        train_batch_size: int = 8,
        val_batch_size: int = 8,
        patch_size: Union[int, Sequence[int]] = (256, 256),
        num_workers: int = 0,
        pin_memory: bool = False,
        **kwargs,
    ):
        super().__init__()

        self.data_dir = data_path
        self.train_batch_size = train_batch_size
        self.val_batch_size = val_batch_size
        self.patch_size = patch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory

    def setup(self, stage: Optional[str] = None) -> None:

#       =========================  VQ-VAE Dataset  =========================
        train_transforms = transforms.Compose(
            [
                transforms.ToImage(),
                transforms.ToDtype(torch.float32, scale=True),
                transforms.Lambda(
                    lambda x: F.interpolate(
                        x.unsqueeze(0), size=(64, 64), mode="area"
                    ).squeeze(0)
                ),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ]
        )

        val_transforms = transforms.Compose(
            [
                transforms.ToImage(),
                transforms.ToDtype(torch.float32, scale=True),
                transforms.Lambda(
                    lambda x: F.interpolate(
                        x.unsqueeze(0), size=(64, 64), mode="area"
                    ).squeeze(0)
                ),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ]
        )


        self.train_dataset = MultiAgentPOVDataset(
            h5_path=self.data_dir, split="train", val_ratio=0.2, transform=train_transforms
        )

        self.val_dataset = MultiAgentPOVDataset(
            h5_path=self.data_dir, split="val", val_ratio=0.2, transform=val_transforms
        )
#       ===============================================================
        
    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.train_batch_size,
            num_workers=self.num_workers,
            shuffle=True,
            pin_memory=self.pin_memory,
        )

    def val_dataloader(self) -> Union[DataLoader, List[DataLoader]]:
        return DataLoader(
            self.val_dataset,
            batch_size=self.val_batch_size,
            num_workers=self.num_workers,
            shuffle=False,
            pin_memory=self.pin_memory,
        )
    
    def test_dataloader(self) -> Union[DataLoader, List[DataLoader]]:
        return DataLoader(
            self.val_dataset,
            batch_size=144,
            num_workers=self.num_workers,
            shuffle=True,
            pin_memory=self.pin_memory,
        )
     

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()